# Telco Customer Churn — Exploratory Data Analysis (EDA)

**Project Overview:**  
This notebook performs an end-to-end exploratory data analysis on the IBM Telco Customer Churn dataset. The goal is to uncover patterns and key factors that drive customers to leave, answering the core business question: *"Who is at risk of churning, and what causes it?"*

| Info | Detail |
|------|--------|
| **Dataset** | IBM Telco Customer Churn |
| **Records** | 7,043 customers, 21 features |
| **Target** | `Churn` (Yes / No) |
| **Goal** | Exploratory analysis to understand churn behavior |

---

## Table of Contents
1. [Setup & Libraries](#1)
2. [Data Loading & Cleaning](#2)
3. [Target Distribution (Churn)](#3)
4. [Demographic Analysis](#4)
5. [Internet & Phone Services Analysis](#5)
6. [Contract & Payment Analysis](#6)
7. [Financial Analysis (Charges)](#7)
8. [Correlation & Heatmap](#8)
9. [Summary & Business Insights](#9)

---
<a id='1'></a>
## 1. Setup & Libraries

Import all libraries required for data analysis and visualization.

In [28]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
%matplotlib inline

# Global plot configuration for clean, professional visuals
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style='whitegrid', palette='muted')

print("✅ All libraries imported successfully!")
print(f"   - pandas   : {pd.__version__}")
print(f"   - numpy    : {np.__version__}")
print(f"   - seaborn  : {sns.__version__}")

---
<a id='2'></a>
## 2. Data Loading & Cleaning

Before analysis, we ensure the data is clean and all column types are correct.
Notably, `TotalCharges` is loaded as a string and must be converted to numeric.

In [ ]:
# --- Load Data ---
df = pd.read_csv('telco_churn.csv')

print(f"Initial shape: {df.shape} (rows, columns)")
print(f"\n--- Data Types ---")
print(df.dtypes)

# --- Cleaning: Convert TotalCharges to numeric ---
# Some rows have whitespace strings (customers with tenure=0), which cause NaN after conversion
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

null_mask = df['TotalCharges'].isnull()
print(f"\n  {null_mask.sum()} rows with empty TotalCharges (tenure = 0 months)")
print(df[null_mask][['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']])

# Fill NaN with 0 (new customers have no accumulated charges yet)
df['TotalCharges'].fillna(0, inplace=True)

# # --- Cleaning: Drop non-analytical column ---
df.drop(columns=['customerID'], inplace=True)

# --- Cleaning: Create binary churn column (1 = churned, 0 = retained) ---
df['Churn_Binary'] = df['Churn'].map({'Yes': 1, 'No': 0})

print(f"\n✅ Data after cleaning: {df.shape} (rows, columns)")
print(f"   Remaining missing values: {df.isnull().sum().sum()}")

Initial shape: (7043, 21) (rows, columns)

--- Data Types ---
customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

  11 rows with empty TotalCharges (tenure = 0 months)
      customerID  tenure  MonthlyCharges  TotalCharges
488   4472-LVYGI       0           52.55           NaN
753   3115-CZMZD       0           20.25           NaN
936   5709-LVOEQ       0           80.85           NaN
1082  4367-NUYAO       0           25.75           NaN
1340  

In [32]:
df.to_csv('telco_churn_clean.csv', index=False)

In [30]:
# Descriptive statistics for numeric columns
print("--- Descriptive Statistics ---")
df[['tenure', 'MonthlyCharges', 'TotalCharges']].describe().round(2)

---
<a id='3'></a>
## 3. Target Distribution: Churn vs Retained

The first step in any EDA is examining the **target variable** distribution.
This reveals whether the dataset is imbalanced and how to approach modeling.

In [20]:
churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Customer Distribution: Churned vs Retained', fontsize=16, fontweight='bold', y=1.01)

# --- Bar chart ---
colors = ['#2ecc71', '#e74c3c']
bars = axes[0].bar(churn_counts.index, churn_counts.values, color=colors, edgecolor='white', width=0.5)
axes[0].set_title('Customer Count', fontsize=13)
axes[0].set_xlabel('Churn Status')
axes[0].set_ylabel('Number of Customers')
for bar, count, pct in zip(bars, churn_counts.values, churn_pct.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f'{count:,}\n({pct:.1f}%)', ha='center', fontsize=12, fontweight='bold')

# --- Donut chart ---
wedge_props = dict(width=0.5, edgecolor='white', linewidth=3)
axes[1].pie(
    churn_counts.values, labels=churn_counts.index,
    colors=colors, autopct='%1.1f%%', startangle=90,
    wedgeprops=wedge_props, textprops={'fontsize': 13}
)
axes[1].set_title('Churn Proportion', fontsize=13)

plt.tight_layout()
plt.savefig('eda_01_churn_distribution.png', bbox_inches='tight', dpi=150)
plt.show()

print(f"📊 Summary:")
print(f"   Retained : {churn_counts['No']:,} customers ({churn_pct['No']:.1f}%)")
print(f"   Churned  : {churn_counts['Yes']:,} customers ({churn_pct['Yes']:.1f}%)")

---
<a id='4'></a>
## 4. Demographic Analysis

Examining whether demographic factors — **gender**, **senior citizen status**, **partner**, and **dependents** — influence churn behavior.

In [21]:
demo_cols = ['gender', 'SeniorCitizen', 'Partner', 'Dependents']
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle('Churn Rate by Demographic Factors', fontsize=16, fontweight='bold')

for i, col in enumerate(demo_cols):
    ax = axes[i]
    if col == 'SeniorCitizen':
        # Map 0/1 to meaningful labels
        temp = df.copy()
        temp[col] = temp[col].map({0: 'Non-Senior', 1: 'Senior'})
    else:
        temp = df

    # Calculate churn rate per category
    churn_rate = temp.groupby(col)['Churn_Binary'].mean().reset_index()
    churn_rate['Churn_Rate_%'] = churn_rate['Churn_Binary'] * 100

    bars = ax.bar(
        churn_rate[col], churn_rate['Churn_Rate_%'],
        color=['#2ecc71', '#e74c3c'][:len(churn_rate)],
        edgecolor='white', width=0.5
    )
    ax.set_title(col, fontsize=13, fontweight='bold')
    ax.set_ylabel('Churn Rate (%)' if i == 0 else '')
    ax.set_ylim(0, 65)

    for bar, val in zip(bars, churn_rate['Churn_Rate_%']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{val:.1f}%', ha='center', fontsize=12, fontweight='bold', color='#2c3e50')

plt.tight_layout()
plt.savefig('eda_02_demographics.png', bbox_inches='tight', dpi=150)
plt.show()

---
<a id='5'></a>
## 5. Internet & Phone Services Analysis

How does the type of service a customer subscribes to relate to their likelihood of churning?
Add-on services like `OnlineSecurity` and `TechSupport` may act as retention drivers.

In [22]:
service_cols = ['PhoneService', 'MultipleLines', 'InternetService',
                'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
fig.suptitle('Churn Rate by Service Type', fontsize=17, fontweight='bold', y=1.01)
axes = axes.flatten()

color_map = {
    'No': '#2ecc71',
    'Yes': '#e74c3c',
    'No phone service': '#95a5a6',
    'No internet service': '#95a5a6',
    'DSL': '#3498db',
    'Fiber optic': '#e67e22'
}
default_colors = ['#3498db', '#e74c3c', '#95a5a6']

for i, col in enumerate(service_cols):
    ax = axes[i]
    churn_rate = df.groupby(col)['Churn_Binary'].mean().reset_index()
    churn_rate['Churn_Rate_%'] = churn_rate['Churn_Binary'] * 100
    churn_rate = churn_rate.sort_values('Churn_Rate_%', ascending=False)

    colors = [color_map.get(v, default_colors[j % len(default_colors)])
              for j, v in enumerate(churn_rate[col])]

    bars = ax.bar(churn_rate[col], churn_rate['Churn_Rate_%'],
                  color=colors, edgecolor='white', width=0.5)
    ax.set_title(col, fontsize=12, fontweight='bold')
    ax.set_ylabel('Churn Rate (%)' if i % 3 == 0 else '')
    ax.set_ylim(0, 80)
    ax.tick_params(axis='x', rotation=15)

    for bar, val in zip(bars, churn_rate['Churn_Rate_%']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{val:.1f}%', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('eda_03_services.png', bbox_inches='tight', dpi=150)
plt.show()

---
<a id='6'></a>
## 6. Contract & Payment Analysis

Contract type and payment method are strong indicators of customer loyalty.
Month-to-month customers face no switching cost, making them the highest-risk group.

In [23]:
contract_cols = ['Contract', 'PaperlessBilling', 'PaymentMethod']

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Churn Rate by Contract & Payment Method', fontsize=16, fontweight='bold')

contract_colors = ['#e74c3c', '#f39c12', '#2ecc71']
payment_colors  = ['#e74c3c', '#3498db', '#9b59b6', '#1abc9c']
billing_colors  = ['#2ecc71', '#e74c3c']

for i, col in enumerate(contract_cols):
    ax = axes[i]
    churn_rate = df.groupby(col)['Churn_Binary'].mean().reset_index()
    churn_rate['Churn_Rate_%'] = churn_rate['Churn_Binary'] * 100
    churn_rate = churn_rate.sort_values('Churn_Rate_%', ascending=False)

    colors = contract_colors if col == 'Contract' else (payment_colors if col == 'PaymentMethod' else billing_colors)
    colors = colors[:len(churn_rate)]

    bars = ax.barh(churn_rate[col], churn_rate['Churn_Rate_%'],
                   color=colors, edgecolor='white', height=0.5)
    ax.set_title(col, fontsize=13, fontweight='bold')
    ax.set_xlabel('Churn Rate (%)')
    ax.set_xlim(0, 60)

    for bar, val in zip(bars, churn_rate['Churn_Rate_%']):
        ax.text(val + 0.5, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}%', va='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('eda_04_contract_payment.png', bbox_inches='tight', dpi=150)
plt.show()

---
<a id='7'></a>
## 7. Financial Analysis: Tenure, Monthly & Total Charges

Numeric variables — how long a customer has been subscribed and how much they pay —
are often the strongest predictors of churn.

In [24]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
churn_yes = df[df['Churn'] == 'Yes']
churn_no  = df[df['Churn'] == 'No']

fig, axes = plt.subplots(3, 2, figsize=(16, 14))
fig.suptitle('Distribution & Boxplot of Numeric Variables: Churned vs Retained',
             fontsize=15, fontweight='bold')

for i, col in enumerate(num_cols):
    # --- Histogram (distribution) ---
    ax_hist = axes[i][0]
    ax_hist.hist(churn_no[col], bins=40, alpha=0.7, color='#2ecc71', label='Retained', edgecolor='white')
    ax_hist.hist(churn_yes[col], bins=40, alpha=0.7, color='#e74c3c', label='Churned', edgecolor='white')
    ax_hist.set_title(f'{col} Distribution', fontsize=12, fontweight='bold')
    ax_hist.set_xlabel(col)
    ax_hist.set_ylabel('Frequency')
    ax_hist.legend(fontsize=10)

    # --- Boxplot ---
    ax_box = axes[i][1]
    data_to_plot = [churn_no[col].values, churn_yes[col].values]
    bp = ax_box.boxplot(data_to_plot, patch_artist=True,
                        labels=['Retained', 'Churned'],
                        medianprops=dict(color='white', linewidth=2.5))
    bp['boxes'][0].set_facecolor('#2ecc71')
    bp['boxes'][1].set_facecolor('#e74c3c')
    ax_box.set_title(f'{col} Boxplot', fontsize=12, fontweight='bold')
    ax_box.set_ylabel(col)

    for j, d in enumerate(data_to_plot):
        median_val = np.median(d)
        ax_box.text(j+1, median_val + (max(d)*0.02), f'Med: {median_val:.0f}',
                    ha='center', fontsize=10, color='#2c3e50', fontweight='bold')

plt.tight_layout()
plt.savefig('eda_05_financial.png', bbox_inches='tight', dpi=150)
plt.show()

print("📊 Median per group:")
for col in num_cols:
    m_no  = df[df['Churn']=='No'][col].median()
    m_yes = df[df['Churn']=='Yes'][col].median()
    print(f"   {col:20s}: Retained = {m_no:.1f} | Churned = {m_yes:.1f}")

---
<a id='8'></a>
## 8. Correlation & Heatmap

A correlation heatmap reveals the **linear relationships** between all numeric variables at once.
This helps identify which features are most associated with churn.

In [ ]:
# Encode binary categorical columns (Yes/No -> 1/0) for correlation computation
df_encoded = df.copy()
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService',
               'PaperlessBilling', 'Churn']
for col in binary_cols:
    df_encoded[col] = df_encoded[col].map({'Yes': 1, 'No': 0,
                                            'Male': 1, 'Female': 0})

# Select numeric columns only
numeric_df = df_encoded[['gender', 'SeniorCitizen', 'Partner', 'Dependents',
                          'tenure', 'PhoneService', 'PaperlessBilling',
                          'MonthlyCharges', 'TotalCharges', 'Churn_Binary']]

corr_matrix = numeric_df.corr()

fig, ax = plt.subplots(figsize=(12, 9))
cmap = sns.diverging_palette(220, 10, as_cmap=True)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # Show lower triangle only

sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap=cmap, center=0, vmin=-1, vmax=1,
    linewidths=0.5, linecolor='white',
    square=True, ax=ax, cbar_kws={'shrink': 0.8}
)
ax.set_title('Correlation Heatmap of Numeric Features', fontsize=15, fontweight='bold', pad=15)

plt.tight_layout()
plt.savefig('eda_06_correlation.png', bbox_inches='tight', dpi=150)
plt.show()

# Show correlations with Churn, ranked by absolute value
print("📊 Correlation with Churn (ranked by strength):")
churn_corr = corr_matrix['Churn_Binary'].drop('Churn_Binary').sort_values(key=abs, ascending=False)
for feat, val in churn_corr.items():
    direction = '⬆️ positive' if val > 0 else '⬇️ negative'
    print(f"   {feat:20s}: {val:+.3f}  ({direction})")

---
<a id='9'></a>
## 9. Summary & Business Insights

After a thorough exploration, we consolidate the key findings into actionable business insights.

In [ ]:
# ============================================================
# SUMMARY: Key Findings from Exploratory Data Analysis
# ============================================================

print("=" * 60)
print("   EDA SUMMARY — Telco Customer Churn")
print("=" * 60)

# 1. Overall churn rate
churn_rate_overall = df['Churn_Binary'].mean() * 100
print(f"🔹 1. Overall Churn Rate      : {churn_rate_overall:.1f}%")

# 2. Senior citizen churn
senior_churn    = df[df['SeniorCitizen']==1]['Churn_Binary'].mean() * 100
non_senior_churn = df[df['SeniorCitizen']==0]['Churn_Binary'].mean() * 100
print(f"🔹 2. Senior Citizen Churn    : {senior_churn:.1f}%  |  Non-Senior : {non_senior_churn:.1f}%")

# 3. Internet service churn rate
for svc in df['InternetService'].unique():
    rate = df[df['InternetService']==svc]['Churn_Binary'].mean() * 100
    print(f"🔹 3. Churn Rate ({svc:12s}) : {rate:.1f}%")

# 4. Contract type churn rate
for contract in ['Month-to-month', 'One year', 'Two year']:
    rate = df[df['Contract']==contract]['Churn_Binary'].mean() * 100
    print(f"🔹 4. Churn Rate ({contract:18s}) : {rate:.1f}%")

# 5. Median tenure by churn group
med_tenure_churn   = df[df['Churn']=='Yes']['tenure'].median()
med_tenure_nochurn = df[df['Churn']=='No']['tenure'].median()
print(f"🔹 5. Median Tenure  | Churned: {med_tenure_churn:.0f} mo | Retained: {med_tenure_nochurn:.0f} mo")

# 6. Median monthly charges by churn group
med_mc_churn   = df[df['Churn']=='Yes']['MonthlyCharges'].median()
med_mc_nochurn = df[df['Churn']=='No']['MonthlyCharges'].median()
print(f"🔹 6. Median Monthly | Churned: ${med_mc_churn:.2f} | Retained: ${med_mc_nochurn:.2f}")

print("=" * 60)

## Executive Summary & Business Strategy

Beyond standard descriptive statistics, this analysis uncovers the latent variables driving customer attrition and formulates actionable retention architectures:

### 1. Product Asymmetry (Demographics vs. Infrastructure)
* **Finding:** Senior Citizens have a 2x higher churn probability, and Fiber Optic services contribute to the highest attrition rate (42%).
* **Diagnosis:** A technical literacy gap. Seniors on Fiber Optic networks are highly susceptible to churn when facing minor technical issues due to a lack of automated or prioritized support.
* **Recommendation:** Deploy automated, zero-cost `TechSupport` routing specifically for the senior demographic on Fiber Optic plans to prevent operational panic and subsequent service cancellation.

### 2. Redefining Loyalty (Survivorship Bias & Trailing Indicators)
* **Finding:** Correlation matrices show high `tenure` is inversely related to churn risk.
* **Diagnosis:** `Tenure` is a trailing indicator, not a causal metric. The near-zero churn in high-tenure cohorts is a manifestation of *Survivorship Bias* — they remained because their local network infrastructure happened to be stable. Passive customers who continue paying are trapped in *Customer Inertia*, not necessarily loyal.
* **Recommendation:** Cease relying on `tenure` as a safety metric. Shift focus to proactive Quality of Service (QoS) monitoring in regions showing high first-month churn spikes.

### 3. Financial Retention Architecture (LTV vs. ARPU)
* **Finding:** Month-to-month (MTM) customers generate the highest median revenue ($79 to $111+) but possess the maximum churn probability (43%). 
* **Diagnosis:** Offering cash discounts to retain these high-risk entities is a financial flaw that destroys Average Revenue Per User (ARPU).
* **Recommendation:** Execute *Ecosystem Lock-in*. Mandate an upgrade to a 1-year contract, compensated purely through the injection of Value-Added Services (VAS) such as priority tech support or streaming bundles. This sacrifices marginal Customer Acquisition Cost (CAC) to secure massive Customer Lifetime Value (LTV) cash flow.(Online Security, Tech Support) are strongly associated with lower churn rates
